In [1]:
import os
import shutil
import pandas as pd

# --- CONFIG ---
# Thay đổi đường dẫn này trỏ tới thư mục chứa 4 folder class của bạn
INPUT_DIR = '/kaggle/input/datasets/zehrakucuker/brain-tumor-mri-images-classification-dataset/dataset' 
OUTPUT_IMG_DIR = '/kaggle/working/images'
OUTPUT_CSV = '/kaggle/working/brain_tumor_mri_metadata.csv'

# --- DATA PREP ---
def prepare_dataset():
    # Tạo folder output nếu chưa tồn tại
    os.makedirs(OUTPUT_IMG_DIR, exist_ok=True)
    
    metadata_list = []
    
    # Lấy danh sách các folder (mỗi folder là 1 class)
    class_names = sorted([d for d in os.listdir(INPUT_DIR) if os.path.isdir(os.path.join(INPUT_DIR, d))])
    
    print(f"Found classes: {class_names}")
    
    for class_name in class_names:
        class_path = os.path.join(INPUT_DIR, class_name)
        
        # Lấy danh sách ảnh và sắp xếp lại
        img_list = sorted([f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        
        # Dùng enumerate để sinh số thứ tự (i) từ 0
        for i, img_name in enumerate(img_list):
            
            # 1. Rename: "tên folder"_"số thứ tự 4 chữ số".jpg
            # i:04d sẽ biến số 1 thành 0001, 15 thành 0015
            new_img_name = f"{class_name}_{i:04d}.jpg"
            
            old_path = os.path.join(class_path, img_name)
            new_path = os.path.join(OUTPUT_IMG_DIR, new_img_name)
            
            # 2. Copy file
            import shutil
            shutil.copy2(old_path, new_path)
            
            # 3. Add to list
            metadata_list.append({
                'filename': new_img_name,
                'class_label': class_name
            })
            
    # --- METADATA PROCESSING ---
    df = pd.DataFrame(metadata_list)
    
    # 4. One-hot Encode
    # pandas get_dummies sẽ tự động biến cột class_label thành N cột 0/1
    df = pd.get_dummies(df, columns=['class_label'], dtype=int)
    
    # Xóa chữ "class_label_" ở tiêu đề cột cho gọn
    df.columns = df.columns.str.replace('class_label_', '')
    
    # 5. Shuffle Data
    # frac=1 nghĩa là lấy 100% dữ liệu, trộn đều và reset lại index
    df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # 6. Export to CSV
    df_shuffled.to_csv(OUTPUT_CSV, index=False)
    
    print(f"Successfully processed {len(df_shuffled)} images.")
    return df_shuffled

# --- MAIN ---
if __name__ == "__main__":
    df_meta = prepare_dataset()
    print("\nMetadata Preview:")
    print(df_meta.head(5))

Found classes: ['glioma', 'healthy', 'meningioma', 'pituitary']
Successfully processed 15605 images.

Metadata Preview:
              filename  glioma  healthy  meningioma  pituitary
0     healthy_0996.jpg       0        1           0          0
1   pituitary_0822.jpg       0        0           0          1
2  meningioma_1191.jpg       0        0           1          0
3     healthy_3874.jpg       0        1           0          0
4      glioma_3708.jpg       1        0           0          0


In [2]:
shutil.make_archive("brain_tumor_mri_images", 'zip', "/kaggle/working/images")

'/kaggle/working/brain_tumor_mri_images.zip'

In [3]:
from tqdm.auto import tqdm
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from PIL import Image

# --- CONFIG ---
CSV_PATH = '/kaggle/working/brain_tumor_mri_metadata.csv'
IMG_DIR = '/kaggle/working/images'
BATCH_SIZE = 128
IMG_SIZE = 224 # Chuẩn đầu vào của EfficientNet-B0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. DATASET: ĐƯA TOÀN BỘ ẢNH LÊN RAM ---
class BrainTumorRAMDataset(Dataset):
    def __init__(self, df, img_dir, classes, transform=None):
        self.transform = transform
        self.images = []
        self.labels = []
        
        print(f"Đang nạp {len(df)} ảnh vào bộ nhớ RAM. Vui lòng đợi (chỉ mất 1 lần)...")
        for idx in tqdm(range(len(df))):
            row = df.iloc[idx]
            img_path = os.path.join(img_dir, row['filename'])
            
            # Đọc ảnh và ngay lập tức Resize về 224x224 để giảm tải cho CPU sau này
            # Lưu ý: Convert sang RGB là chuẩn chung cho mô hình phân loại
            image = Image.open(img_path).convert('RGB')
            image = image.resize((IMG_SIZE, IMG_SIZE))
            
            # Giải mã One-hot thành số nguyên
            one_hot = row[classes].values.astype(np.float32)
            class_idx = np.argmax(one_hot)
            
            self.images.append(image)
            self.labels.append(class_idx)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Lúc train, chỉ việc bốc ảnh từ RAM ra, cực kỳ nhanh!
        image = self.images[idx]
        label = self.labels[idx]
        
        # Chỉ thực hiện các phép biến đổi nhẹ nhàng (Flip, ToTensor) trên on-the-fly
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)
# --- SPLIT & LOADERS ---
# Đọc file CSV
df = pd.read_csv(CSV_PATH)
classes = ['glioma', 'healthy', 'meningioma', 'pituitary']

# Tạo cột label tạm thời để dùng cho stratify
df['temp_label'] = df[classes].idxmax(axis=1)

# Chia 70-20-10
# 1. Tách Test (10%)
df_train_val, df_test = train_test_split(df, test_size=0.1, random_state=42, stratify=df['temp_label'])
# 2. Tách Train (70%) và Val (20%) từ 90% còn lại. Tỉ lệ Val = 2/9
df_train, df_val = train_test_split(df_train_val, test_size=(2.0/9.0), random_state=42, stratify=df_train_val['temp_label'])

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

# --- 2. TRANSFORMS (ĐÃ TỐI ƯU) ---
# Transform chuẩn của ImageNet cho EfficientNet
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(), # Data Augmentation nhẹ
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 3. KHỞI TẠO LẠI DATASETS & LOADERS ---
# Quá trình này sẽ mất khoảng 1-2 phút để đẩy data lên RAM
print("\n[PREPARING TRAIN DATA]")
train_dataset = BrainTumorRAMDataset(df_train, IMG_DIR, classes, transform=train_transform)

print("\n[PREPARING VAL DATA]")
val_dataset = BrainTumorRAMDataset(df_val, IMG_DIR, classes, transform=test_transform)

print("\n[PREPARING TEST DATA]")
test_dataset = BrainTumorRAMDataset(df_test, IMG_DIR, classes, transform=test_transform)

# DataLoader giờ đây lấy thẳng từ RAM, nên num_workers=2 hoặc 4 sẽ chạy mượt mà
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

Train: 10923 | Val: 3121 | Test: 1561

[PREPARING TRAIN DATA]
Đang nạp 10923 ảnh vào bộ nhớ RAM. Vui lòng đợi (chỉ mất 1 lần)...


  0%|          | 0/10923 [00:00<?, ?it/s]


[PREPARING VAL DATA]
Đang nạp 3121 ảnh vào bộ nhớ RAM. Vui lòng đợi (chỉ mất 1 lần)...


  0%|          | 0/3121 [00:00<?, ?it/s]


[PREPARING TEST DATA]
Đang nạp 1561 ảnh vào bộ nhớ RAM. Vui lòng đợi (chỉ mất 1 lần)...


  0%|          | 0/1561 [00:00<?, ?it/s]

In [4]:
## TRAIN WITH RUNTIME

import time # Thêm thư viện time
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from tqdm.auto import tqdm

# --- CONFIG TRAIN ---
EPOCHS = 50
PATIENCE = 7
LR = 1e-3
LOG_FILE = "training_log.csv"
BEST_MODEL_PATH = "best_efficientnet_b0.pt"

# --- INIT MODEL ---
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, len(classes))

# KÍCH HOẠT MULTI-GPU (DataParallel)
if torch.cuda.device_count() > 1:
    print(f"Tuyệt vời! Đang sử dụng {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scaler = torch.GradScaler()

# --- TRAINING PIPELINE ---
def train_efficientnet():
    history = {'epoch': [], 'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    best_val_loss = float('inf')
    patience_counter = 0
    
    # ⏱️ BẮT ĐẦU BẤM GIỜ
    start_time = time.time()
    print("[INFO] Bắt đầu quá trình huấn luyện...")
    
    for epoch in range(EPOCHS):
        # 1. TRAIN
        model.train()
        train_loss, correct, total = 0.0, 0, 0
        
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [TRAIN]")
        for images, labels in loop:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            
            with torch.autocast(device_type='cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            loop.set_postfix(loss=loss.item())
            
        avg_train_loss = train_loss / total
        train_acc = correct / total
        
        # 2. VALIDATION
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [VAL]"):
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                
                with torch.autocast(device_type='cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
        avg_val_loss = val_loss / total
        val_acc = correct / total
        
        print(f"Epoch {epoch+1}/{EPOCHS} [TRAIN]")
        print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        # 3. LOGGING
        history['epoch'].append(epoch + 1)
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)
        pd.DataFrame(history).to_csv(LOG_FILE, index=False)
        
        # 4. EARLY STOPPING & CHECKPOINT
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            
            state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(state_dict, BEST_MODEL_PATH)
            
            print(" -> [SAVED BEST MODEL]")
        else:
            patience_counter += 1
            print(f" -> No improvement. Patience: {patience_counter}/{PATIENCE}")
            if patience_counter >= PATIENCE:
                print(" -> [EARLY STOPPING TRIGGERED]")
                break

    # ⏱️ KẾT THÚC BẤM GIỜ & TÍNH TOÁN RUNTIME
    end_time = time.time()
    total_time = end_time - start_time
    
    # Quy đổi ra Phút và Giây cho dễ đọc
    mins = int(total_time // 60)
    secs = int(total_time % 60)
    print(f"\n=========================================")
    print(f"🎉 HOÀN TẤT HUẤN LUYỆN!")
    print(f"⏱️ Tổng thời gian chạy (Runtime): {mins} phút {secs} giây")
    print(f"=========================================\n")

# Chạy huấn luyện
if __name__ == "__main__":
    train_efficientnet()

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 140MB/s] 


Tuyệt vời! Đang sử dụng 2 GPUs!
[INFO] Bắt đầu quá trình huấn luyện...


Epoch 1/50 [TRAIN]:   0%|          | 0/86 [00:00<?, ?it/s]

Epoch 1/50 [VAL]:   0%|          | 0/25 [00:00<?, ?it/s]

...

Epoch 17/50 [VAL]:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 17/50 [TRAIN]
Train Loss: 0.0135 | Train Acc: 0.9962
Val Loss: 0.0147 | Val Acc: 0.9981
 -> No improvement. Patience: 6/7


Epoch 18/50 [TRAIN]:   0%|          | 0/86 [00:00<?, ?it/s]

Epoch 18/50 [VAL]:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 18/50 [TRAIN]
Train Loss: 0.0121 | Train Acc: 0.9962
Val Loss: 0.0199 | Val Acc: 0.9981
 -> No improvement. Patience: 7/7
 -> [EARLY STOPPING TRIGGERED]

🎉 HOÀN TẤT HUẤN LUYỆN!
⏱️ Tổng thời gian chạy (Runtime): 132 phút 18 giây



In [5]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# --- TEST EVALUATION ---
def test_and_evaluate():
    print(f"Loading {BEST_MODEL_PATH} for testing...")
    model.load_state_dict(torch.load(BEST_MODEL_PATH))
    model.eval()
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Testing"):
            images = images.to(DEVICE)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    # 1. METRICS (Dùng 'macro' để tính trung bình công bằng cho tất cả 4 classes)
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    
    print("\n--- TEST METRICS ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print("--------------------\n")
    
    # 2. CONFUSION MATRICES
    cm = confusion_matrix(all_labels, all_preds)
    cm_norm = confusion_matrix(all_labels, all_preds, normalize='true')
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Absolute CM
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=classes, yticklabels=classes)
    axes[0].set_xlabel('Predicted Label')
    axes[0].set_ylabel('True Label')
    axes[0].set_title('Absolute Confusion Matrix')
    
    # Normalized CM
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=axes[1],
                xticklabels=classes, yticklabels=classes)
    axes[1].set_xlabel('Predicted Label')
    axes[1].set_ylabel('True Label')
    axes[1].set_title('Normalized Confusion Matrix')
    
    plt.tight_layout()
    plt.show()

# Chạy test
if __name__ == "__main__":
    test_and_evaluate()

Loading best_efficientnet_b0.pt for testing...


RuntimeError: Error(s) in loading state_dict for DataParallel:
	Missing key(s) in state_dict: "module.features.0.0.weight", "module.features.0.1.weight", "module.features.0.1.bias", "module.features.0.1.running_mean", "module.features.0.1.running_var", "module.features.1.0.block.0.0.weight", "module.features.1.0.block.0.1.weight", "module.features.1.0.block.0.1.bias", "module.features.1.0.block.0.1.running_mean", "module.features.1.0.block.0.1.running_var", "module.features.1.0.block.1.fc1.weight", "module.features.1.0.block.1.fc1.bias", "module.features.1.0.block.1.fc2.weight", "module.features.1.0.block.1.fc2.bias", "module.features.1.0.block.2.0.weight", "module.features.1.0.block.2.1.weight", "module.features.1.0.block.2.1.bias", "module.features.1.0.block.2.1.running_mean", "module.features.1.0.block.2.1.running_var", "module.features.2.0.block.0.0.weight", "module.features.2.0.block.0.1.weight", "module.features.2.0.block.0.1.bias", "module.features.2.0.block.0.1.running_mean", "